# External Event Triggers

This demo shows how to control megane's viewer from external tools (e.g., Plotly) via events.

- **Frame selection**: Click a Plotly data point to switch trajectory frames
- **Atom selection & measurement**: Select atoms from Python and retrieve distances, angles, and dihedrals
- **Event callbacks**: Subscribe to megane events to update other widgets


In [ ]:
import megane

print(f"megane v{megane.__version__}")

## 1. Plotly Integration: Click to Select Frame

Click a data point on the time-series plot (e.g., energy) to display
the corresponding trajectory frame in the viewer.


In [ ]:
import numpy as np

pipe = megane.Pipeline()
s = pipe.add_node(megane.LoadStructure("../tests/fixtures/caffeine_water.pdb"))
t = pipe.add_node(megane.LoadTrajectory(xtc="../tests/fixtures/caffeine_water_vibration.xtc"))
bonds = pipe.add_node(megane.AddBonds(source="structure"))
vp = pipe.add_node(megane.Viewport())
pipe.add_edge(s.out.particle, t.inp.particle)
pipe.add_edge(s.out.particle, bonds.inp.particle)
pipe.add_edge(s.out.particle, vp.inp.particle)
pipe.add_edge(t.out.traj, vp.inp.traj)
pipe.add_edge(bonds.out.bond, vp.inp.bond)

viewer = megane.MolecularViewer()
viewer.set_pipeline(pipe)
print(f"Loaded trajectory: {viewer.total_frames} frames")

In [ ]:
import plotly.graph_objects as go
import ipywidgets as widgets

# Generate dummy energy data
n_frames = viewer.total_frames
frames = np.arange(n_frames)
energy = -500 + 10 * np.sin(frames * 0.1) + np.random.randn(n_frames) * 2

# Create Plotly FigureWidget
fig = go.FigureWidget(
    data=[go.Scatter(x=frames, y=energy, mode="lines+markers", name="Energy")],
    layout=go.Layout(
        title="Energy vs Frame (click a point to jump)",
        xaxis_title="Frame",
        yaxis_title="Energy (kJ/mol)",
        height=300,
    ),
)

# Connect click event to megane frame selection
def on_plotly_click(trace, points, state):
    if points.point_inds:
        viewer.frame_index = points.point_inds[0]

fig.data[0].on_click(on_plotly_click)

# Display stacked vertically
widgets.VBox([fig, viewer])

## 2. Programmatic Atom Selection and Measurement

Set `selected_atoms` to a list of atom indices to highlight atoms
in the viewer and automatically compute measurements.

| Atoms | Measurement |
|-------|-------------|
| 2 | Distance (Å) |
| 3 | Angle (°) |
| 4 | Dihedral (°) |


In [ ]:
v_pipe = megane.Pipeline()
s = v_pipe.add_node(megane.LoadStructure("../tests/fixtures/caffeine_water.pdb"))
bonds = v_pipe.add_node(megane.AddBonds(source="distance"))
vp = v_pipe.add_node(megane.Viewport())
v_pipe.add_edge(s.out.particle, bonds.inp.particle)
v_pipe.add_edge(s.out.particle, vp.inp.particle)
v_pipe.add_edge(bonds.out.bond, vp.inp.bond)

v = megane.MolecularViewer()
v.set_pipeline(v_pipe)
v

In [ ]:
# 2 atoms → distance
v.selected_atoms = [0, 1]
print("Distance:", v.measurement)

In [ ]:
# 3 atoms → angle
v.selected_atoms = [0, 1, 2]
print("Angle:", v.measurement)

In [ ]:
# 4 atoms → dihedral angle
v.selected_atoms = [0, 1, 2, 3]
print("Dihedral:", v.measurement)

In [ ]:
# Clear selection
v.selected_atoms = []

## 3. Event Callbacks

Use `on_event()` to subscribe to megane events.
Works as both a decorator and a method call.

| Event | Trigger | Data |
|-------|---------|------|
| `frame_change` | `frame_index` changes | `{"frame_index": int}` |
| `selection_change` | `selected_atoms` changes | `{"atoms": [int, ...]}` |
| `measurement` | Measurement result updated | `{"type", "value", "label", "atoms"}` |


In [ ]:
cb_pipe = megane.Pipeline()
s = cb_pipe.add_node(megane.LoadStructure("../tests/fixtures/caffeine_water.pdb"))
t = cb_pipe.add_node(megane.LoadTrajectory(xtc="../tests/fixtures/caffeine_water_vibration.xtc"))
bonds = cb_pipe.add_node(megane.AddBonds(source="structure"))
vp = cb_pipe.add_node(megane.Viewport())
cb_pipe.add_edge(s.out.particle, t.inp.particle)
cb_pipe.add_edge(s.out.particle, bonds.inp.particle)
cb_pipe.add_edge(s.out.particle, vp.inp.particle)
cb_pipe.add_edge(t.out.traj, vp.inp.traj)
cb_pipe.add_edge(bonds.out.bond, vp.inp.bond)

cb_viewer = megane.MolecularViewer()
cb_viewer.set_pipeline(cb_pipe)

# Register events with decorators
@cb_viewer.on_event("frame_change")
def on_frame(data):
    print(f"Frame changed: {data['frame_index']}")

@cb_viewer.on_event("measurement")
def on_meas(data):
    if data:
        print(f"Measurement: {data['type']} = {data['label']}")

cb_viewer

In [ ]:
# Change frame from Python → frame_change event fires
cb_viewer.frame_index = 10

In [ ]:
# Unregister callback
cb_viewer.off_event("frame_change", on_frame)
cb_viewer.frame_index = 20  # no longer prints

## 4. Bidirectional Sync: Update Plotly Marker on Frame Change

Use megane's frame change event to synchronize
a marker on the Plotly chart with the current frame position.


In [ ]:
sync_pipe = megane.Pipeline()
s = sync_pipe.add_node(megane.LoadStructure("../tests/fixtures/caffeine_water.pdb"))
t = sync_pipe.add_node(megane.LoadTrajectory(xtc="../tests/fixtures/caffeine_water_vibration.xtc"))
bonds = sync_pipe.add_node(megane.AddBonds(source="structure"))
vp = sync_pipe.add_node(megane.Viewport())
sync_pipe.add_edge(s.out.particle, t.inp.particle)
sync_pipe.add_edge(s.out.particle, bonds.inp.particle)
sync_pipe.add_edge(s.out.particle, vp.inp.particle)
sync_pipe.add_edge(t.out.traj, vp.inp.traj)
sync_pipe.add_edge(bonds.out.bond, vp.inp.bond)

sync_viewer = megane.MolecularViewer()
sync_viewer.set_pipeline(sync_pipe)

n = sync_viewer.total_frames
x = np.arange(n)
y = -500 + 10 * np.sin(x * 0.1) + np.random.randn(n) * 2

sync_fig = go.FigureWidget(
    data=[
        go.Scatter(x=x, y=y, mode="lines", name="Energy"),
        go.Scatter(x=[0], y=[y[0]], mode="markers", marker=dict(size=12, color="red"), name="Current"),
    ],
    layout=go.Layout(title="Bidirectional sync", height=300,
                     xaxis_title="Frame", yaxis_title="Energy (kJ/mol)"),
)

# Plotly → megane
def on_click(trace, points, state):
    if points.point_inds:
        sync_viewer.frame_index = points.point_inds[0]

sync_fig.data[0].on_click(on_click)

# megane → Plotly (update red marker)
@sync_viewer.on_event("frame_change")
def update_marker(data):
    idx = data["frame_index"]
    with sync_fig.batch_update():
        sync_fig.data[1].x = [idx]
        sync_fig.data[1].y = [y[idx]]

widgets.VBox([sync_fig, sync_viewer])

## 5. Dihedral Angle Trajectory Analysis

Compute the dihedral angle at each frame and plot the results with Plotly.
Combines `frame_index` and `selected_atoms`.


In [ ]:
import numpy as np
from megane.parsers.xtc import load_trajectory

dih_pipe = megane.Pipeline()
s = dih_pipe.add_node(megane.LoadStructure("../tests/fixtures/caffeine_water.pdb"))
t = dih_pipe.add_node(megane.LoadTrajectory(xtc="../tests/fixtures/caffeine_water_vibration.xtc"))
bonds = dih_pipe.add_node(megane.AddBonds(source="structure"))
vp = dih_pipe.add_node(megane.Viewport())
dih_pipe.add_edge(s.out.particle, t.inp.particle)
dih_pipe.add_edge(s.out.particle, bonds.inp.particle)
dih_pipe.add_edge(s.out.particle, vp.inp.particle)
dih_pipe.add_edge(t.out.traj, vp.inp.traj)
dih_pipe.add_edge(bonds.out.bond, vp.inp.bond)

dih_viewer = megane.MolecularViewer()
dih_viewer.set_pipeline(dih_pipe)

# Highlight the four atoms in the live viewer (browser computes the
# instantaneous measurement on every frame change).
atoms = [0, 1, 2, 3]
dih_viewer.selected_atoms = atoms


def dihedral_deg(p0, p1, p2, p3):
    """Dihedral angle (degrees) between planes (p0,p1,p2) and (p1,p2,p3)."""
    b0 = p0 - p1
    b1 = p2 - p1
    b2 = p3 - p2
    b1n = b1 / np.linalg.norm(b1)
    v = b0 - np.dot(b0, b1n) * b1n
    w = b2 - np.dot(b2, b1n) * b1n
    return float(np.degrees(np.arctan2(np.dot(np.cross(b1n, v), w), np.dot(v, w))))


# Compute the dihedral series directly from trajectory frames.
# A Python `for` loop that only sets `frame_index` would NOT receive
# `measurement` events from the JS frontend, because the kernel cannot
# process incoming comm messages while the loop is running.
trajectory = load_trajectory(
    "../tests/fixtures/caffeine_water.pdb",
    "../tests/fixtures/caffeine_water_vibration.xtc",
)
dihedrals = [
    dihedral_deg(*[trajectory.get_frame(i)[a] for a in atoms])
    for i in range(trajectory.n_frames)
]
assert len(dihedrals) == dih_viewer.total_frames, (
    f"expected {dih_viewer.total_frames} dihedrals, got {len(dihedrals)}"
)

dih_fig = go.FigureWidget(
    data=[go.Scatter(x=list(range(len(dihedrals))), y=dihedrals, mode="lines+markers")],
    layout=go.Layout(
        title="Dihedral angle along trajectory",
        xaxis_title="Frame",
        yaxis_title="Dihedral (°)",
        height=300,
    ),
)

widgets.VBox([dih_fig, dih_viewer])

## 6. Custom Widget Integration

Connect any widget (e.g., `ipywidgets` sliders) with `on_event` for seamless integration.


In [ ]:
slider_pipe = megane.Pipeline()
s = slider_pipe.add_node(megane.LoadStructure("../tests/fixtures/caffeine_water.pdb"))
t = slider_pipe.add_node(megane.LoadTrajectory(xtc="../tests/fixtures/caffeine_water_vibration.xtc"))
bonds = slider_pipe.add_node(megane.AddBonds(source="structure"))
vp = slider_pipe.add_node(megane.Viewport())
slider_pipe.add_edge(s.out.particle, t.inp.particle)
slider_pipe.add_edge(s.out.particle, bonds.inp.particle)
slider_pipe.add_edge(s.out.particle, vp.inp.particle)
slider_pipe.add_edge(t.out.traj, vp.inp.traj)
slider_pipe.add_edge(bonds.out.bond, vp.inp.bond)

slider_viewer = megane.MolecularViewer()
slider_viewer.set_pipeline(slider_pipe)

# Slider/label mirror megane's built-in playback bar, which displays
# `currentFrame + 1 / totalFrames` (1-based). Keeping the same convention
# avoids the off-by-one disagreement users would otherwise see between
# the slider/label ("Frame: 53") and the viewer footer ("54 / 100").
n_frames = slider_viewer.total_frames
slider = widgets.IntSlider(
    value=1, min=1, max=n_frames,
    description="Frame:", continuous_update=True,
)
label = widgets.Label(value=f"Frame: 1 / {n_frames}")

# Slider → megane: convert 1-based display to 0-based frame_index.
def on_slider(change):
    slider_viewer.frame_index = change["new"] - 1

slider.observe(on_slider, names="value")

# megane → label/slider: format as 1-based to match megane's footer.
@slider_viewer.on_event("frame_change")
def on_frame_for_label(data):
    idx = data["frame_index"]  # 0-based internally
    label.value = f"Frame: {idx + 1} / {n_frames}"
    slider.value = idx + 1

# Sanity check: setting frame_index 5 (0-based) must surface as "6 / N"
# in both label and slider, matching what the megane footer would show.
slider_viewer.frame_index = 5
assert slider.value == 6, f"slider should be 6 (1-based), got {slider.value}"
assert label.value == f"Frame: 6 / {n_frames}", label.value
slider_viewer.frame_index = 0  # reset for display

widgets.VBox([widgets.HBox([slider, label]), slider_viewer])